In [1]:
import pandas as pd

CLEANED = "../Progress/code/datasets/cleaned"
MAPPING = "../Progress/code/datasets/mapping"

# Load
masters = pd.read_csv(f"{CLEANED}/masters_cleaned_for_mapping.csv")

print(f"Shape: {masters.shape}")
print(f"Columns: {masters.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(masters.head().to_string())

Shape: (57085, 6)
Columns: ['program_name', 'program_type', 'program_name_clean', 'degree_text', 'degree_norm', 'remove_flag']

First 5 rows:
                                    program_name program_type                             program_name_clean                                    degree_text                                    degree_norm  remove_flag
0                                      Economics          MSc                                      economics                                      economics                                      economics        False
1    Political Science and International Affairs       Master    political science and international affairs    political science and international affairs    political science and international affairs        False
2                        Business Administration          MBA                        business administration                        business administration                        business administration        

In [2]:
# How many unique programs and degree labels
print(f"Total rows: {masters.shape[0]}")
print(f"Unique program_name: {masters['program_name'].nunique()}")
print(f"Unique degree_norm: {masters['degree_norm'].nunique()}")
print(f"Unique program_type: {masters['program_type'].nunique()}")

print(f"\nProgram types distribution:")
print(masters['program_type'].value_counts().head(20).to_string())

print(f"\n200 sample degree_norm values (sorted):")
print(
    masters['degree_norm']
    .dropna()
    .drop_duplicates()
    .sample(200, random_state=42)
    .sort_values()
    .reset_index(drop=True)
    .to_string()
)

Total rows: 57085
Unique program_name: 23780
Unique degree_norm: 23356
Unique program_type: 46

Program types distribution:
program_type
MSc                    24710
MA                     13015
Master                  9998
MBA                     2661
MEd                     1697
LLM                     1051
MEng                     987
MPhil                    692
MRes                     584
Pre-Master               457
MFA                      185
MM                       116
MLitt                    102
MPA                       95
MPH                       77
MAT                       63
MSW                       62
Master of Music           51
MSN                       36
Master of Fine Arts       32

200 sample degree_norm values (sorted):
0                            21st century media practice
1                                 accountancy coursework
2                                 accounting and banking
3                                        adult education
4      adult e

In [3]:
# Keyword-based broad field classifier
# Order matters — first match wins, so put specific before general

field_rules = [
    # Computing & Technology
    ("Computing & Technology", [
        "computer science", "computing", "software", "information technology",
        "information systems", "data science", "artificial intelligence",
        "machine learning", "cybersecurity", "network", "human computer",
        "web", "digital systems", "electronic engineering", "electronics",
        "telecommunication", "wireless", "distributed systems", "informatics",
        "computer engineering", "it management", "technology management",
        "innovation technology", "advanced computer", "internet"
    ]),

    # Engineering
    ("Engineering", [
        "engineering", "mechanical", "civil engineering", "structural",
        "aerospace", "chemical engineering", "electrical engineering",
        "industrial engineering", "manufacturing", "robotics", "automotive",
        "naval architecture", "systems engineering", "process technology",
        "safety engineering", "railway", "building services"
    ]),

    # Business & Management
    ("Business & Management", [
        "business administration", "business management", "mba",
        "management", "entrepreneurship", "operations management",
        "project management", "supply chain", "logistics", "procurement",
        "organizational", "corporate", "strategic management",
        "international business", "business strategy", "business research",
        "business innovation", "retail management"
    ]),

    # Finance & Economics
    ("Finance & Economics", [
        "finance", "economics", "econometrics", "accounting", "banking",
        "financial", "investment", "actuarial", "risk management",
        "financial risk", "quantitative finance", "wealth management",
        "cost estimating", "taxation"
    ]),

    # Health & Medicine
    ("Health & Medicine", [
        "nursing", "medicine", "medical", "clinical", "pharmacy",
        "pharmacology", "public health", "health science", "healthcare",
        "health management", "dentistry", "dental", "nutrition",
        "dietetics", "physiotherapy", "occupational therapy",
        "speech", "optometry", "radiology", "pathology", "epidemiology",
        "global health", "health studies", "health promotion",
        "physician assistant", "midwifery", "mental health",
        "health informatics", "biomedical", "oral sciences",
        "advanced practice", "prescribing", "medicines"
    ]),

    # Life Sciences & Biology
    ("Life Sciences & Biology", [
        "biology", "biochemistry", "biotechnology", "genetics",
        "microbiology", "ecology", "evolutionary", "marine biology",
        "zoology", "botany", "genomics", "molecular biology",
        "cell biology", "neuroscience", "life sciences",
        "biodiversity", "conservation biology", "synthetic biology",
        "biological", "agricultural science", "food science",
        "crop", "aquatic", "comparative medicine"
    ]),

    # Physical & Mathematical Sciences
    ("Physical & Mathematical Sciences", [
        "physics", "chemistry", "mathematics", "statistics",
        "applied mathematics", "mathematical", "astrophysics",
        "astronomy", "materials science", "nanomaterials",
        "nanotechnology", "geological", "geophysics", "earth science",
        "physical science", "chemical", "pharmaceutical chemistry",
        "stochastic", "geodetic", "radioecology", "photonics"
    ]),

    # Environmental & Earth Sciences
    ("Environmental Sciences", [
        "environmental", "sustainability", "climate", "energy",
        "renewable energy", "green energy", "ecology", "geography",
        "urban planning", "urban design", "water", "flood",
        "waste management", "natural resources", "forest",
        "wildlife", "ocean", "marine", "earth", "resource efficiency",
        "sustainable", "nature management"
    ]),

    # Law & Policy
    ("Law & Policy", [
        "law", "legal", "llm", "jurisprudence", "criminology",
        "public policy", "policy", "governance", "public administration",
        "international relations", "political science", "diplomacy",
        "human rights", "security studies", "terrorism",
        "intelligence", "homeland security", "emergency management",
        "public management"
    ]),

    # Education
    ("Education", [
        "education", "teaching", "pedagogy", "curriculum",
        "instructional", "learning technology", "literacy",
        "special education", "early childhood", "elementary",
        "secondary education", "higher education", "teacher",
        "vocational education", "educational", "academic"
    ]),

    # Social Sciences
    ("Social Sciences", [
        "sociology", "social science", "social work", "anthropology",
        "psychology", "social psychology", "community",
        "cultural studies", "area studies", "development studies",
        "international development", "gender studies", "migration",
        "human dimensions", "social research", "criminology",
        "demography", "social policy", "nonprofit", "social foundations",
        "south asian", "european studies", "russian", "latin"
    ]),

    # Arts, Design & Media
    ("Arts, Design & Media", [
        "art", "design", "architecture", "media", "film",
        "photography", "music", "creative", "performing arts",
        "theatre", "dance", "fashion", "fine art", "illustration",
        "animation", "game design", "interior design", "graphic",
        "communication arts", "publishing", "journalism",
        "digital media", "broadcasting", "documentary",
        "museum", "heritage", "arts administration",
        "entertainment", "footwear", "sacred music"
    ]),

    # Humanities
    ("Humanities", [
        "history", "philosophy", "literature", "linguistics",
        "languages", "english", "classics", "religion", "theology",
        "ethics", "cultural", "humanities", "medieval",
        "ancient", "modern languages", "world christianity",
        "portuguese", "french", "italian", "chinese culture"
    ]),

    # Catch-all
    ("Other", [])
]

def classify_field(degree_norm):
    if pd.isna(degree_norm):
        return "Other"
    text = str(degree_norm).lower().strip()
    for field, keywords in field_rules:
        for kw in keywords:
            if kw in text:
                return field
    return "Other"

# Apply to masters dataset
masters["broad_field"] = masters["degree_norm"].apply(classify_field)

# Check distribution
print("Broad field distribution:")
dist = masters["broad_field"].value_counts()
print(dist.to_string())
print(f"\nTotal classified: {len(masters)}")
print(f"Unclassified (Other): {(masters['broad_field'] == 'Other').sum()}")
print(f"Coverage: {((masters['broad_field'] != 'Other').sum() / len(masters) * 100):.1f}%")

Broad field distribution:
broad_field
Other                               11710
Business & Management                7355
Arts, Design & Media                 4555
Engineering                          4350
Education                            4159
Health & Medicine                    4140
Computing & Technology               3317
Humanities                           2943
Law & Policy                         2805
Finance & Economics                  2776
Physical & Mathematical Sciences     2721
Social Sciences                      2592
Life Sciences & Biology              1918
Environmental Sciences               1744

Total classified: 57085
Unclassified (Other): 11710
Coverage: 79.5%


In [4]:
# Sample the unclassified ones to see what we're missing
other = masters[masters["broad_field"] == "Other"]["degree_norm"].dropna()

print(f"Unclassified: {len(other)}")
print(f"Unique unclassified degree_norm: {other.nunique()}")

print("\n100 sample unclassified degree_norm values:")
print(
    other.drop_duplicates()
    .sample(min(100, other.nunique()), random_state=42)
    .sort_values()
    .reset_index(drop=True)
    .to_string()
)

# Also check top 30 most frequent unclassified
print("\nTop 30 most frequent unclassified:")
print(other.value_counts().head(30).to_string())

Unclassified: 11710
Unique unclassified degree_norm: 5536

100 sample unclassified degree_norm values:
0                                      accelerated path
1                       aerodynamics and aerostructures
2     aists advanced studies mas in sport administra...
3                       analysing language use research
4       animal behaviour: applications for conservation
5                                   applied gerontology
6                                           aquaculture
7       assistive technology studies and human services
8                               athletic administration
9                                      bible exposition
10                                         chiropractic
11    city and metropolitan planning: ecological pla...
12                           city and regional planning
13                                    classical singing
14                               coaching and mentoring
15                             cognition and creativity
1

In [5]:
# Additional keywords to catch the unclassified
extra_rules = {
    "Business & Management": [
        "marketing", "executive", "business analytics", "real estate",
        "international marketing", "public relations", "strategy and innovation",
        "leadership", "administration", "retail", "hospitality",
        "event management", "sport administration", "athletic administration",
        "recreation", "leisure", "coaching and mentoring", "labour policies",
        "enterprise", "commerce"
    ],
    "Computing & Technology": [
        "cyber security", "information processing", "signal and information",
        "pattern recognition", "intelligent system", "statistical learning",
        "neurotechnology", "nanoscience", "quantum", "controlled quantum",
        "lifespan digital", "digital culture", "science in information",
        "inverse and ill posed", "precision machinery", "semiconductor"
    ],
    "Health & Medicine": [
        "health administration", "health care", "kinesiology", "exercise science",
        "athletic training", "rehabilitation", "counseling", "school counseling",
        "gerontology", "ageing", "chiropractic", "ultrasound", "diagnostic imaging",
        "surgical", "prosthodontics", "orthodontics", "paramedic",
        "animal behaviour", "veterinary", "laboratory animal",
        "women's health", "family nurse", "health care practice",
        "applied gerontology", "movement leisure", "assistive technology"
    ],
    "Social Sciences": [
        "communication", "communication studies", "communication science",
        "criminal justice", "criminology", "conflict analysis",
        "american studies", "international studies", "cultural studies",
        "culture and society", "digital culture", "indigenous",
        "labour", "social service", "gender", "justice",
        "pacific and asian", "irish studies", "women's studies",
        "holocaust", "cognition and creativity", "religious studies",
        "bible", "missional", "world christianity", "defence",
        "planning", "city and regional", "urban", "spatial"
    ],
    "Life Sciences & Biology": [
        "animal science", "livestock", "aquaculture", "kinesiology",
        "exercise science", "veterinary", "laboratory animal",
        "animal behaviour", "quaternary science", "gerontology",
        "natural sciences", "science studies", "applied gerontology"
    ],
    "Physical & Mathematical Sciences": [
        "geology", "geosciences", "nanoscience", "quantum",
        "tribology", "aerodynamics", "aerostructures",
        "pure and applied logic", "statistical learning",
        "signal and information", "inverse and ill posed",
        "mathematical physics", "quaternary", "forensics science",
        "forensic science"
    ],
    "Environmental Sciences": [
        "planning", "city planning", "regional planning",
        "fire protection", "livestock health", "aquaculture",
        "ecological", "geosciences"
    ],
    "Education": [
        "school counseling", "moderate special needs", "special needs",
        "sign language", "writing for children", "writing for young",
        "liberal studies", "professional studies", "postgraduate diploma",
        "accelerated path", "preparation", "science thesis",
        "research center", "graduate diploma"
    ],
    "Arts, Design & Media": [
        "composition", "classical singing", "double bass", "jazz studies",
        "writing for", "textile", "apparel", "spatial design",
        "theaterwissenschaft", "technical heritage", "tpti"
    ],
    "Humanities": [
        "spanish", "japanese", "german", "cognitive science",
        "cognitive", "language use", "analysing language",
        "american studies", "pacific and asian", "archaeology",
        "religious studies", "bible exposition", "holocaust",
        "missional", "irish studies", "labour policies"
    ],
    "Law & Policy": [
        "criminal justice", "conflict analysis", "justice",
        "defence", "labour policies", "indigenous development",
        "international graduate studies"
    ],
    "Finance & Economics": [
        "business analytics", "real estate", "international marketing",
        "marketing research", "marketing and innovation"
    ],
}

def classify_field_v2(degree_norm):
    if pd.isna(degree_norm):
        return "Other"
    text = str(degree_norm).lower().strip()
    
    # First pass — original rules
    for field, keywords in field_rules:
        for kw in keywords:
            if kw in text:
                return field
    
    # Second pass — extra rules
    for field, keywords in extra_rules.items():
        for kw in keywords:
            if kw in text:
                return field
    
    return "Other"

# Apply v2
masters["broad_field"] = masters["degree_norm"].apply(classify_field_v2)

print("Broad field distribution (v2):")
dist = masters["broad_field"].value_counts()
print(dist.to_string())
print(f"\nCoverage: {((masters['broad_field'] != 'Other').sum() / len(masters) * 100):.1f}%")
print(f"Still unclassified: {(masters['broad_field'] == 'Other').sum()}")

# What's still in Other?
still_other = masters[masters["broad_field"] == "Other"]["degree_norm"].dropna()
print(f"\nTop 20 still unclassified:")
print(still_other.value_counts().head(20).to_string())

Broad field distribution (v2):
broad_field
Business & Management               8679
Other                               7431
Health & Medicine                   5007
Arts, Design & Media                4653
Engineering                         4350
Education                           4241
Social Sciences                     3798
Computing & Technology              3422
Humanities                          3268
Physical & Mathematical Sciences    2917
Law & Policy                        2805
Finance & Economics                 2776
Life Sciences & Biology             1988
Environmental Sciences              1750

Coverage: 87.0%
Still unclassified: 7431

Top 20 still unclassified:
degree_norm
business                           43
science                            35
pharmaceutical sciences            30
divinity                           29
marriage and family therapy        28
politics                           28
tesol                              27
applied behavior analysis          

In [6]:
extra_rules_2 = {
    "Business & Management": [
        "business", "administration", "management studies",
        "international", "executive", "commerce", "retail",
        "hospitality", "tourism management", "sport management",
        "recreation management", "facilities management",
        "nonprofit", "public management"
    ],
    "Health & Medicine": [
        "health", "pharmaceutical sciences", "pharmacy sciences",
        "marriage and family therapy", "applied behavior analysis",
        "exercise physiology", "physiology", "kinesiology",
        "occupational", "speech language", "audiology",
        "nutrition", "dietetics", "public health", "epidemiology",
        "biostatistics", "health promotion", "health policy",
        "rehabilitation", "physical therapy", "athletic",
        "family therapy", "behavior analysis", "counseling psychology"
    ],
    "Social Sciences": [
        "politics", "political", "international affairs",
        "interdisciplinary studies", "asian studies",
        "translation studies", "tesol", "applied linguistics",
        "cultural", "sociology", "anthropology", "gender",
        "development", "globalization", "migration",
        "peace studies", "conflict", "human rights",
        "public affairs", "urban affairs", "social"
    ],
    "Humanities": [
        "divinity", "theology", "ministry", "pastoral",
        "reading", "tesol", "translation", "applied linguistics",
        "literature", "writing", "creative writing", "acting",
        "liberal arts", "interdisciplinary", "asian studies",
        "classical", "medieval", "ancient", "modern history"
    ],
    "Life Sciences & Biology": [
        "entomology", "physiology", "exercise physiology",
        "pharmaceutical sciences", "plant science", "soil science",
        "animal science", "food science", "nutrition science",
        "marine science", "environmental science",
        "bioinformatics", "computational biology"
    ],
    "Physical & Mathematical Sciences": [
        "science", "applied science", "pharmaceutical sciences",
        "materials", "optics", "acoustics", "thermodynamics"
    ],
    "Law & Policy": [
        "international affairs", "public affairs", "politics",
        "political", "diplomacy", "international relations",
        "global studies", "global affairs", "security",
        "intelligence studies", "peace studies"
    ],
    "Education": [
        "reading", "tesol", "library and information science",
        "library science", "information science", "instructional",
        "curriculum", "school", "teaching", "literacy",
        "applied behavior analysis", "special needs"
    ],
    "Arts, Design & Media": [
        "acting", "theatre", "film", "creative writing",
        "writing", "music", "fine arts", "visual arts",
        "studio art", "photography", "fashion", "interior",
        "industrial design", "product design", "game design"
    ],
    "Finance & Economics": [
        "finance", "economics", "accounting", "banking",
        "financial", "investment", "actuarial", "taxation",
        "insurance", "risk", "quantitative"
    ],
}

def classify_field_v3(degree_norm):
    if pd.isna(degree_norm):
        return "Other"
    text = str(degree_norm).lower().strip()

    # Pass 1 — original rules
    for field, keywords in field_rules:
        for kw in keywords:
            if kw in text:
                return field

    # Pass 2 — first extra rules
    for field, keywords in extra_rules.items():
        for kw in keywords:
            if kw in text:
                return field

    # Pass 3 — second extra rules
    for field, keywords in extra_rules_2.items():
        for kw in keywords:
            if kw in text:
                return field

    return "Other"

masters["broad_field"] = masters["degree_norm"].apply(classify_field_v3)

print("Broad field distribution (v3):")
print(masters["broad_field"].value_counts().to_string())
print(f"\nCoverage: {((masters['broad_field'] != 'Other').sum() / len(masters) * 100):.1f}%")
print(f"Still unclassified: {(masters['broad_field'] == 'Other').sum()}")

print("\nTop 20 still unclassified:")
print(
    masters[masters["broad_field"] == "Other"]["degree_norm"]
    .dropna()
    .value_counts()
    .head(20)
    .to_string()
)

Broad field distribution (v3):
broad_field
Business & Management               9180
Health & Medicine                   5436
Arts, Design & Media                4654
Social Sciences                     4534
Other                               4422
Engineering                         4350
Education                           4259
Physical & Mathematical Sciences    3652
Humanities                          3617
Computing & Technology              3422
Law & Policy                        2934
Finance & Economics                 2807
Life Sciences & Biology             2068
Environmental Sciences              1750

Coverage: 92.3%
Still unclassified: 4422

Top 20 still unclassified:
degree_norm
toxicology                  20
professional accountancy    20
directing                   19
environment                 18
analytics                   18
safety                      18
technology                  18
historic preservation       18
theological studies         17
innovation            

In [7]:
# Final pass — catch remaining high-frequency ones
extra_rules_3 = {
    "Life Sciences & Biology": [
        "toxicology", "agriculture", "nurse anesthesia",
        "entomology", "horticulture", "agronomy", "viticulture"
    ],
    "Finance & Economics": [
        "professional accountancy", "analytics", "data analytics",
        "accounting", "actuarial"
    ],
    "Arts, Design & Media": [
        "directing", "painting", "performance", "ceramics",
        "sculpture", "printmaking", "illustration", "animation",
        "historic preservation", "conservation"
    ],
    "Humanities": [
        "theological studies", "islamic studies", "african studies",
        "divinity", "ministry", "biblical", "religious"
    ],
    "Computing & Technology": [
        "technology", "analytics", "data analytics", "embedded systems",
        "innovation", "safety", "systems"
    ],
    "Environmental Sciences": [
        "environment", "agriculture", "safety", "toxicology"
    ],
    "Business & Management": [
        "tourism", "innovation", "safety management"
    ],
    "Education": [
        "full time", "professional studies"
    ],
}

def classify_field_final(degree_norm):
    if pd.isna(degree_norm):
        return "Interdisciplinary & Other"
    text = str(degree_norm).lower().strip()

    for field, keywords in field_rules:
        for kw in keywords:
            if kw in text:
                return field
    for field, keywords in extra_rules.items():
        for kw in keywords:
            if kw in text:
                return field
    for field, keywords in extra_rules_2.items():
        for kw in keywords:
            if kw in text:
                return field
    for field, keywords in extra_rules_3.items():
        for kw in keywords:
            if kw in text:
                return field

    return "Interdisciplinary & Other"

masters["broad_field"] = masters["degree_norm"].apply(classify_field_final)

print("Final broad field distribution:")
dist = masters["broad_field"].value_counts()
print(dist.to_string())
print(f"\nFinal coverage: {((masters['broad_field'] != 'Interdisciplinary & Other').sum() / len(masters) * 100):.1f}%")
print(f"Interdisciplinary & Other: {(masters['broad_field'] == 'Interdisciplinary & Other').sum()}")

Final broad field distribution:
broad_field
Business & Management               9209
Health & Medicine                   5436
Arts, Design & Media                4900
Social Sciences                     4534
Engineering                         4350
Education                           4280
Computing & Technology              3823
Humanities                          3704
Physical & Mathematical Sciences    3652
Interdisciplinary & Other           3399
Law & Policy                        2934
Finance & Economics                 2903
Life Sciences & Biology             2174
Environmental Sciences              1787

Final coverage: 94.0%
Interdisciplinary & Other: 3399


In [8]:
# Save the enriched masters table with broad_field
masters.to_csv(f"../Progress/code/datasets/cleaned/masters_with_fields.csv", index=False)
print("Saved: masters_with_fields.csv")

# Build the two-level lookup
# Level 1: broad_field
# Level 2: unique program_name + program_type combos within each field

# Join broad_field onto best_enriched output
best_enriched = pd.read_csv(f"../Progress/code/datasets/mapping/program_best_enriched_gcsi.csv")

# Merge field info — use program_name as join key
field_lookup = masters[["program_name", "program_type", "degree_norm", "broad_field"]].drop_duplicates()

best_enriched_full = best_enriched.merge(
    field_lookup,
    on=["program_name", "program_type"],
    how="left"
)

print(f"\nbest_enriched_full shape: {best_enriched_full.shape}")
print(f"Rows with broad_field: {best_enriched_full['broad_field'].notna().sum()}")
print(f"Rows missing broad_field: {best_enriched_full['broad_field'].isna().sum()}")

# Build the dropdown lookup table
# This is what the frontend will use for the two-level dropdown
dropdown = (
    masters[["broad_field", "program_name", "program_type", "degree_norm"]]
    .drop_duplicates(subset=["broad_field", "program_name", "program_type"])
    .sort_values(["broad_field", "program_name"])
    .reset_index(drop=True)
)

print(f"\nDropdown table shape: {dropdown.shape}")
print(f"Unique broad fields: {dropdown['broad_field'].nunique()}")
print(f"Unique programs: {dropdown['program_name'].nunique()}")

print("\nSample — Computing & Technology:")
print(dropdown[dropdown['broad_field'] == 'Computing & Technology'].head(10).to_string())

print("\nSample — Health & Medicine:")
print(dropdown[dropdown['broad_field'] == 'Health & Medicine'].head(10).to_string())

Saved: masters_with_fields.csv

best_enriched_full shape: (57085, 17)
Rows with broad_field: 57085
Rows missing broad_field: 0

Dropdown table shape: (27793, 4)
Unique broad fields: 14
Unique programs: 23780

Sample — Computing & Technology:
                 broad_field                                                         program_name program_type                                                        degree_norm
7301  Computing & Technology                                                    AGSM (Technology)          MBA                                                    agsm technology
7302  Computing & Technology                                  Accounting & Information Technology          MSc                                  accounting information technology
7303  Computing & Technology         Accounting (Concentration in Accounting Information Systems)          MSc         accounting concentration in accounting information systems
7304  Computing & Technology                  

In [9]:
# Save dropdown lookup
dropdown.to_csv("../Progress/code/datasets/mapping/program_field_dropdown.csv", index=False)
print("Saved: program_field_dropdown.csv")

# Save best_enriched with broad_field attached
best_enriched_full.to_csv("../Progress/code/datasets/mapping/program_best_enriched_gcsi.csv", index=False)
print("Saved: program_best_enriched_gcsi.csv (updated with broad_field)")

# Final summary of what we have
print("\n=== FINAL DATASET SUMMARY ===")
print(f"\nDropdown (program_field_dropdown.csv):")
print(f"  {dropdown['broad_field'].nunique()} broad fields")
print(f"  {dropdown['program_name'].nunique()} unique programs")
print(f"  {dropdown['program_type'].nunique()} degree types")

print(f"\nBroad field sizes:")
for field, count in dropdown['broad_field'].value_counts().items():
    print(f"  {field:<35} {count:>5} programs")

print(f"\nRecommendation table (program_best_enriched_gcsi.csv):")
print(f"  {best_enriched_full.shape[0]:,} rows")
print(f"  Columns: {best_enriched_full.columns.tolist()}")

print(f"\nOccupation GCSI table (occupation_gcsi.csv):")
gcsi = pd.read_csv("../Progress/code/datasets/mapping/occupation_gcsi.csv")
print(f"  {gcsi.shape[0]} occupations with GCSI scores")
print(f"  GCSI range: {gcsi['GCSI'].min():.1f} – {gcsi['GCSI'].max():.1f}")
print(f"  Mean GCSI: {gcsi['GCSI'].mean():.1f}")

Saved: program_field_dropdown.csv
Saved: program_best_enriched_gcsi.csv (updated with broad_field)

=== FINAL DATASET SUMMARY ===

Dropdown (program_field_dropdown.csv):
  14 broad fields
  23780 unique programs
  46 degree types

Broad field sizes:
  Business & Management                4609 programs
  Arts, Design & Media                 2692 programs
  Health & Medicine                    2628 programs
  Interdisciplinary & Other            2418 programs
  Education                            2335 programs
  Social Sciences                      2042 programs
  Computing & Technology               1770 programs
  Engineering                          1764 programs
  Humanities                           1644 programs
  Physical & Mathematical Sciences     1478 programs
  Law & Policy                         1438 programs
  Environmental Sciences               1047 programs
  Finance & Economics                   992 programs
  Life Sciences & Biology               936 programs

Recomme

In [10]:
import shutil
import os

# Source and destination
SRC  = "../Progress/code/datasets/mapping"
DEST = "../Final/datasets"

# Files to copy
files = [
    "occupation_gcsi.csv",
    "program_best_enriched_gcsi.csv",
    "program_field_dropdown.csv",
    "occupation_skill_bridge_onet.csv",
    "occupation_technology_skills_onet_summary.csv",
]

# Create destination folder if it doesn't exist
os.makedirs(DEST, exist_ok=True)

# Copy each file
for f in files:
    src_path  = f"{SRC}/{f}"
    dest_path = f"{DEST}/{f}"
    shutil.copy2(src_path, dest_path)
    size = os.path.getsize(dest_path) / (1024 * 1024)
    print(f"✓ {f} ({size:.1f} MB)")

print(f"\nAll files copied to {DEST}")
print(f"Files in Final/datasets:")
for f in os.listdir(DEST):
    size = os.path.getsize(f"{DEST}/{f}") / (1024 * 1024)
    print(f"  {f} ({size:.1f} MB)")

✓ occupation_gcsi.csv (0.1 MB)
✓ program_best_enriched_gcsi.csv (11.8 MB)
✓ program_field_dropdown.csv (2.4 MB)
✓ occupation_skill_bridge_onet.csv (12.6 MB)
✓ occupation_technology_skills_onet_summary.csv (0.1 MB)

All files copied to ../Final/datasets
Files in Final/datasets:
  occupation_technology_skills_onet_summary.csv (0.1 MB)
  program_field_dropdown.csv (2.4 MB)
  occupation_gcsi.csv (0.1 MB)
  occupation_skill_bridge_onet.csv (12.6 MB)
  program_best_enriched_gcsi.csv (11.8 MB)


In [11]:
import os

files = [
    "occupation_gcsi.csv",
    "program_best_enriched_gcsi.csv",
    "program_field_dropdown.csv",
    "occupation_skill_bridge_onet.csv",
    "occupation_technology_skills_onet_summary.csv",
]

print("File sizes:")
total = 0
for f in files:
    path = f"datasets/{f}"
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        total += size_mb
        print(f"  {f:<55} {size_mb:>7.1f} MB")
    else:
        print(f"  {f:<55} NOT FOUND")
print(f"\nTotal: {total:.1f} MB")

File sizes:
  occupation_gcsi.csv                                         0.1 MB
  program_best_enriched_gcsi.csv                             11.8 MB
  program_field_dropdown.csv                                  2.4 MB
  occupation_skill_bridge_onet.csv                           12.6 MB
  occupation_technology_skills_onet_summary.csv               0.1 MB

Total: 26.9 MB
